# Hand in 7 

## Subscriber

## In general:

**IoT layers:**
The different computing layers are: edge, fog and cloud. *Edge* is the sensors that collect the data, *fog* is where the sensor data is stored at first (could be a smartphone or another device with computing power) and the *cloud* is a collection of remote servers.


**mqtt:**
*Message queuing telemetry transport* is an application protocol that uses publishers and subscribers in order to send diffent data "topics" across devices.

In [3]:
import paho.mqtt.client as mqtt
import pandas as pd
import ast
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

data=pd. DataFrame([]) # DataFrame we use to store the information we receive
count=0 # Tracker to tell how far we are in the dataset
correct_prediction=0

# This function defines what to do when we connect to the broker
def on_connect(client, userdata, flags, rc):
    print("Connected with result code " + str(rc))
    # We subscribe to this particular data. There may be other data published by the same subscriber or broker.
    client.subscribe("diabetes_data")


# This function defines what to do when we receives a message from the brokder
def on_message(client, userdata, msg):
    msg_dict=pd.DataFrame([ast.literal_eval(msg.payload.decode())]) #decoding msg to string, string to dictionary and dictionary to pd object
    
    global data,count,correct_prediction # taking global object
    
    if msg_dict is not None:
        data = pd.concat([data, msg_dict], ignore_index=True) 
        count+=1
        print(count)
        l=len(data)
        if(l>5):
            train_d=data[-6:-1] # Slice the current dataset to get 5 last entries
            test_d=data[l-1:l+1] # Get the newest entry 
            
            X_train=train_d[['Glucose','BloodPressure','Insulin']]
            y_train=train_d['Outcome']
            
            X_test=test_d[[ 'Glucose','BloodPressure','Insulin']]
            y_test=test_d['Outcome'].values[0] # The label of our test data
            
            knn = KNeighborsClassifier(n_neighbors=3)
            knn.fit(X_train, y_train)
            
            y_pred_test = knn.predict(X_test)
            
            if(y_pred_test==y_test):
                correct_prediction+=1
                print('Current number of correct classifications:',correct_prediction)
        if(count==1000):
            print('After 1000 times The Final number of correct classifications',correct_prediction)

    
mqttc = mqtt.Client()
# We create a client as the data subscriber and specify its actions for particular events
mqttc.on_connect = on_connect
mqttc.on_message = on_message
# Now, we connect to the data broker.
mqttc.connect("mqtt.eclipseprojects.io", 1883, 60)
# As a simple example, we just keep the data listening/receiving on and on...

mqttc.loop_start()
if count>=1000:

    mqttc.disconnect()
    mqttc.loop_stop()

Connected with result code 0
1
2
3
4
5
6
Current number of correct classifications: 1
7
8
Current number of correct classifications: 2
9
Current number of correct classifications: 3
10
11
Current number of correct classifications: 4
12
Current number of correct classifications: 5
13
Current number of correct classifications: 6
14
Current number of correct classifications: 7
15
16
Current number of correct classifications: 8
17
Current number of correct classifications: 9
18
19
20
Current number of correct classifications: 10
21
22
23
24
25
Current number of correct classifications: 11
26
27
Current number of correct classifications: 12
28
29
30
Current number of correct classifications: 13
31
Current number of correct classifications: 14
32
33
Current number of correct classifications: 15
34
Current number of correct classifications: 16
35
Current number of correct classifications: 17
36
Current number of correct classifications: 18
37
Current number of correct classifications: 19
38
3

In [4]:
data

""
